In [1]:
import os
import pandas as pd

from quant.config.universe import PRICE_ASSETS, YIELD_ASSETS, FILTER_ASSETS
from quant.data.panels import build_market_panel
from quant.analytics.returns import prices_to_returns, prices_to_diffs

os.chdir("..")

In [14]:
px_last = pd.read_parquet("data/raw/market_panel_px_last.parquet").sort_index()
px_last.head()

ticker,spx,ndx,rty,nikkei,ftse,us2y,us10y,uk10y,bund10y,ig_credit,hy_credit,vix,dxy,eurusd,gbpusd,usdjpy,audusd,oil,gold,ust_intermediate
2007-01-01,NaN,NaN,NaN,NaN,NaN,4.8080,4.7022,4.741,NaN,NaN,NaN,NaN,NaN,1.3201,1.9592,119.04,0.7894,NaN,636.8,NaN
2007-01-02,NaN,NaN,NaN,NaN,6310.9,4.7913,4.6802,4.758,181.842,NaN,NaN,NaN,83.27,1.3273,1.9736,118.85,0.7961,61.05,640.5,NaN
2007-01-03,1416.60,1759.37,787.436,NaN,6319.0,4.7580,4.6582,4.787,181.541,106.45,NaN,12.04,83.92,1.3169,1.9514,119.39,0.7912,58.32,627.5,82.68
2007-01-04,1418.34,1792.91,789.964,17353.67,6287.0,4.6913,4.6024,4.773,181.933,107.25,NaN,11.51,84.34,1.3084,1.9428,119.05,0.7844,55.59,622.0,82.95
2007-01-05,1409.71,1785.30,775.885,17091.59,6220.1,4.7494,4.6442,4.802,181.162,107.19,NaN,12.14,84.64,1.3002,1.9292,118.63,0.7787,56.31,607.4,82.71


In [3]:
px_last.index = pd.to_datetime(px_last.index, errors="coerce")
px_last = px_last[~px_last.index.isna()].sort_index()

In [11]:
# 1) canonicalise the panel (frequency/alignment)
panel = build_market_panel(
    px_last,
    freq="D",  # keep daily for now
    align="outer",  # outer + ffill is usually best for multi-asset
    fill="ffill",
)


display(panel.prices)

ticker,spx,ndx,rty,nikkei,ftse,us2y,us10y,uk10y,bund10y,ig_credit,hy_credit,vix,dxy,eurusd,gbpusd,usdjpy,audusd,oil,gold,ust_intermediate
2007-01-01,NaN,NaN,NaN,NaN,NaN,4.8080,4.7022,4.741,NaN,NaN,NaN,NaN,NaN,1.3201,1.9592,119.04,0.7894,NaN,636.80,NaN
2007-01-02,NaN,NaN,NaN,NaN,6310.90,4.7913,4.6802,4.758,181.842,NaN,NaN,NaN,83.270,1.3273,1.9736,118.85,0.7961,61.05,640.50,NaN
2007-01-03,1416.60,1759.37,787.436,NaN,6319.00,4.7580,4.6582,4.787,181.541,106.45,NaN,12.04,83.920,1.3169,1.9514,119.39,0.7912,58.32,627.50,82.68
2007-01-04,1418.34,1792.91,789.964,17353.67,6287.00,4.6913,4.6024,4.773,181.933,107.25,NaN,11.51,84.340,1.3084,1.9428,119.05,0.7844,55.59,622.00,82.95
2007-01-05,1409.71,1785.30,775.885,17091.59,6220.10,4.7494,4.6442,4.802,181.162,107.19,NaN,12.14,84.640,1.3002,1.9292,118.63,0.7787,56.31,607.40,82.71
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-25,6932.05,25656.15,2548.081,50407.79,9870.68,3.5014,4.1335,4.507,313.362,110.65,80.64,13.47,97.976,1.1784,1.3521,155.83,0.6704,58.35,4479.42,96.35
2025-12-26,6929.94,25644.39,2534.345,50750.39,9870.68,3.4790,4.1277,4.507,313.407,110.64,80.60,13.60,98.022,1.1772,1.3497,156.57,0.6716,56.74,4533.21,96.44
2025-12-29,6905.74,25525.56,2519.798,50526.92,9866.53,3.4545,4.1102,4.486,314.346,110.79,80.63,14.20,98.037,1.1773,1.3512,156.06,0.6694,58.08,4332.35,96.58
2025-12-30,6896.24,25462.56,2500.586,50339.48,9940.71,3.4484,4.1219,4.498,313.745,110.66,80.71,14.33,98.238,1.1748,1.3468,156.41,0.6695,57.95,4339.49,96.48


In [12]:
prices = panel.prices[PRICE_ASSETS]
yields = panel.prices[YIELD_ASSETS]
filters = panel.prices[FILTER_ASSETS]

# 2) correct transforms by type
price_rets = prices_to_returns(prices, method="simple")  # or "simple"
yield_chg = prices_to_diffs(yields)  # basis point-ish level diffs
filter_lvl = filters  # keep level (e.g. VIX) for regime rules

In [13]:
# 3) save
prices.to_parquet("data/processed/prices.parquet")
yields.to_parquet("data/processed/yields.parquet")
price_rets.to_parquet("data/processed/prices_simple_returns.parquet")
yield_chg.to_parquet("data/processed/yield_changes.parquet")
filter_lvl.to_parquet("data/processed/filters.parquet")